# Test pipeline trên trận mới: video → luồng YES → luồng NO (Colab, v6.3)

Chạy **hệ thống đã train** trên trận đấu mới (không có nhãn, không train gì):

| Bước | Việc làm |
|---|---|
| 1–4 | GPU, cấu hình, mount Drive, lấy code |
| 5 | Video 2 hiệp → npy TimeSformer 768 chiều |
| 6 | Phép ép 768 → **npy 8576 chiều** (phép ép đã học ở notebook train) |
| 7 | **Luồng YES**: model đã train bắt sự kiện + sinh câu bình luận → `results_dense_captioning.json` |
| 7b | Tuỳ chọn: **recall** của luồng YES so với sự kiện FIFA, với NMS 0.4 / 0.5 / 0.6 / 0.7 |
| 8–9 | (bản cũ, ElevenLabs) **mặc định bỏ qua**: chỉ chạy khi `RUN_OLD_ELEVENLABS = True` |
| **10** | **Luồng NO**: kịch bản 2.0 + **LLM `gpt-4o-mini`** viết câu bình luận từ metadata FIFA, bình luận liên tục (`CONTINUOUS = True`) |
| 11 | (Gemini TTS) **mặc định bỏ qua**: chỉ chạy khi `RUN_GEMINI_TTS = True` |
| **12** | **Giọng đọc OpenAI `gpt-4o-mini-tts`** cho cả luồng YES (giọng sự kiện) và luồng NO (giọng dẫn chương trình) → video demo |

**Luồng chính (v5.9): cell 1–7 → 10 → 12.** Luồng NO: `gpt-4o-mini` viết câu (cell 10), `gpt-4o-mini-tts` đọc (cell 12). Không dùng ElevenLabs.

**Chỉ làm video demo** (đã có kết quả luồng YES trên Drive từ lần chạy trước): chạy cell **2, 3, 4, 10, 12**. Không cần GPU.

**Cần có trên Drive** (`MyDrive/soccer_project/`), **để Private**:
- `video_test/<giải>/<mùa>/<trận>/1_224p.mp4` và `2_224p.mp4`: hai hiệp đã cắt bằng `cut_halves.py`.
- `video_test/metadata/<trận>/`: kết quả `fifa_crawl.py` (`facts.csv`, `events.csv`, `match_info.csv`...). Tên thư mục trùng tên thư mục trận.
- Từ notebook train: `sn-models/<MODEL_NAME>/` (checkpoint `spotting/` và `caption/`), file phép ép trong `sn-models/`, weights TimeSformer `*.pdparams`.

**API key OpenAI** (cell 10 và 12, bắt buộc): Colab Secrets (biểu tượng 🔑 bên trái), tên `OPENAI_API_KEY`, bật quyền cho notebook. Không dán key vào cell. Một key dùng cho cả `gpt-4o-mini` (viết câu) và `gpt-4o-mini-tts` (đọc).

Không cần key ElevenLabs. Key Gemini (`GEMINI_API_KEY`) chỉ cần khi đặt `NO_LLM = "gemini"`.

Chọn GPU: `Runtime → Change runtime type → T4 GPU`.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cấu hình

- `TEST_GAMES`: chỉ chạy các trận này (mặc định chỉ Chelsea – Man City). `[]` = mọi trận trên Drive.
- `DEMO_HALF`, `DEMO_START`, `DEMO_END`: đoạn làm demo. Mặc định cả hiệp 1 (`"00:00"`, `""`).
- `MODEL_NAME`, `MAPPER_FILE`: model và phép ép **đi theo cặp**, lấy đúng cặp đã train (notebook train cell 2).
- `NMS_THRESHOLD`, `TOP_K`: chỉ dùng lúc suy luận, đổi không cần train lại. NMS thấp hơn thì báo nhiều sự kiện hơn (ít bỏ sót, dễ báo sai hơn).
- Luồng NO (cell 10): `NO_LLM = "openai"`: `gpt-4o-mini` (`OPENAI_LLM_MODEL`) viết lại từng câu của kịch bản 2.0; câu sai số, sai tên, quá dài hoặc có lời khen chê thì dùng lại khuôn. `LLM_DEMO_ONLY = True`: chỉ gọi LLM cho đoạn demo (`DEMO_HALF`, `DEMO_START`–`DEMO_END`; 10 phút: khoảng 110 câu, 1 phút; cả hiệp 1: khoảng 540 câu, 10 phút); `False`: cả trận (hơn 1000 câu, khoảng 15–20 phút). Giọng đọc OpenAI ở cell 12.
- `RUN_OLD_ELEVENLABS`, `RUN_GEMINI_TTS`: bật lại cell 8–9 (ElevenLabs) và cell 11 (Gemini TTS) của bản cũ. Mặc định tắt.

In [ ]:
GITHUB_REPO = "https://github.com/ThuLinh3009/soccer_project.git"
GIT_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soccer_project"
WORK_DIR = "/content/soccernet-caption-deltalab"   # code
TEST_DATA = "/content/test_data"                   # npy của các trận test
TEST_DIR_NAME = "video_test"                       # thư mục trên Drive chứa video + metadata trận test

TEST_GAMES = ["2025-09-05 - Chelsea 2 - 1 Manchester City"]   # (v6.0) chỉ chạy các trận này; [] = mọi trận trên Drive
MODEL_NAME = "gpt2-video60-mlp"                    # model đã train (sn-models/<MODEL_NAME>)
MAPPER_FILE = "mapper_timesformer_mlp_c4.pt"       # phép ép đã học cùng model đó
FEATURES = "baidu_from_timesformer.npy"
GPT_TYPE = "gpt2"
NMS_THRESHOLD = 0.4                                # test: recall 0,78, precision 0,75, ~110 sự kiện/trận
TOP_K = 3
TS_BATCH = 32

RUN_OLD_ELEVENLABS = False                         # cell 8–9 (bản cũ, ElevenLabs): mặc định không chạy
RUN_GEMINI_TTS = False                             # cell 11 (Gemini TTS): mặc định không chạy, giọng đọc ở cell 12
LLM = "elevenlabs"                                 # (cell 8) "template": không gọi LLM, dùng câu fact gốc
LLM_MODEL = "gemini-2.5-flash"
NO_FLOW_GAMES = ["2026-07-05 - Brazil 1 - 2 Norway"]   # trận chạy luồng NO (tiết kiệm credit); [] = tất cả

# v2 (cell 10–11): Gemini
# danh sách model thử lần lượt: model trước hết quota (gói miễn phí) thì dùng model sau
NO_LLM = "openai"                # cell 10: LLM viết câu luồng NO: "openai" | "gemini" | "template" (không LLM)
OPENAI_LLM_MODEL = "gpt-4o-mini"
LLM_DEMO_ONLY = True            # True: chỉ gọi LLM cho đoạn demo (DEMO_HALF, DEMO_START–DEMO_END); False: cả trận
GEMINI_LLM_MODEL = "gemini-2.5-flash-lite,gemini-3.1-flash-lite,gemini-3.5-flash-lite"   # khi NO_LLM = "gemini"
GEMINI_TTS_MODEL = "gemini-3.8-flash-lite-tts,gemini-2.5-flash-preview-tts,gemini-3.1-flash-tts-preview"
VOICE_YES, VOICE_NO = "Puck", "Charon"             # giọng sự kiện / giọng dẫn chương trình
TTS_MAX_LINES = 30                                 # thử trước 30 câu đầu (tiết kiệm quota). 0 = cả trận
WORDS_PER_SECOND = 2.6     # tốc độ đọc dùng để tính số từ tối đa của mỗi câu; cell 11 in ra tốc độ đo được
TTS_MAX_SPEED = 1.25       # câu đọc dài hơn khoảng trống thì tăng tốc tối đa 1,25 lần (1 = không tăng tốc)
TTS_TIMELINE = "v2"        # "v2": timeline cell 10 | "v1": timeline cell 8 (output/timeline, no_flow_llm.py)
NO_FLOW_V2_GAMES = []      # trận chạy cell 10; [] = tất cả
CONTINUOUS = True          # cell 10: luồng NO nói liên tục, lấp mọi khoảng lặng > 3 giây (v5.5)
FILL_LLM = True            # True: câu lấp khoảng lặng cũng do LLM viết (nhiều request: dùng với LLM_DEMO_ONLY)

# cell 12: giọng đọc + video demo
TTS_ENGINE = "openai"             # "openai": OpenAI TTS (key trong Colab Secrets: OPENAI_API_KEY, trả phí theo ký tự)
                                  # "kokoro": chạy trên Colab, không key, không quota
OPENAI_TTS_MODEL = "gpt-4o-mini-tts"
OPENAI_VOICE_YES = "ash"          # giọng đọc sự kiện (alloy, ash, ballad, coral, echo, fable, onyx, nova, sage, shimmer, verse)
OPENAI_VOICE_NO = "onyx"          # giọng dẫn chương trình
OPENAI_STYLE = "epic"             # (v6.2) cách đọc: "epic" = hào hùng như trận chung kết | "calm" = nhẹ nhàng (bản v5.6)
KOKORO_VOICE_YES = "am_michael"   # giọng đọc sự kiện (a = Mỹ: am_michael, am_fenrir, am_puck, am_adam...)
KOKORO_VOICE_NO = "bm_george"     # giọng dẫn chương trình (b = Anh: bm_george, bm_fable, bm_lewis, bm_daniel)
DEMO_GAMES = []                   # trận làm demo; [] = các trận trong TEST_GAMES
DEMO_HALF = 1                     # hiệp làm demo
DEMO_START, DEMO_END = "00:00", ""        # đoạn demo (phút:giây trong hiệp). "00:00", "" = cả hiệp (v6.0: cả hiệp 1)

## 3. Mount Google Drive, tìm trận test, model và weights

In [ ]:
from google.colab import drive
import os, glob
drive.mount("/content/drive", force_remount=True)

if not os.path.isdir(DRIVE_ROOT):
    found = [p for p in glob.glob("/content/drive/MyDrive/**/soccer_project", recursive=True)
             + glob.glob("/content/drive/Shareddrives/*/soccer_project") if os.path.isdir(p)]
    assert found, f"Không thấy {DRIVE_ROOT}: sửa DRIVE_ROOT ở cell 2"
    DRIVE_ROOT = found[0]

TEST_SRC = f"{DRIVE_ROOT}/{TEST_DIR_NAME}"
DRIVE_MODEL_DIR = f"{DRIVE_ROOT}/sn-models"
MAPPER = f"{DRIVE_MODEL_DIR}/{MAPPER_FILE}"
FEAT_CACHE = f"{TEST_SRC}/features"                # npy 768 chiều lưu lại trên Drive
OUT_YES = f"{TEST_SRC}/output/yes"
OUT_TIMELINE = f"{TEST_SRC}/output/timeline"
weights = sorted(glob.glob(f"{DRIVE_ROOT}/**/*.pdparams", recursive=True))
TS_WEIGHTS = weights[0] if weights else None

videos = sorted(glob.glob(f"{TEST_SRC}/*/*/*/[12]_224p.mp4"))
GAMES = sorted({os.path.relpath(os.path.dirname(v), TEST_SRC) for v in videos})
if TEST_GAMES:  # chỉ các trận trong TEST_GAMES (cell 2)
    skipped = [g for g in GAMES if os.path.basename(g) not in TEST_GAMES]
    GAMES = [g for g in GAMES if os.path.basename(g) in TEST_GAMES]
    for g in skipped:
        print("bỏ qua (không có trong TEST_GAMES):", g)
    missing = [t for t in TEST_GAMES if t not in {os.path.basename(g) for g in GAMES}]
    assert not missing, f"Không thấy video 1_224p.mp4 / 2_224p.mp4 của: {missing}"
print("DRIVE_ROOT:", DRIVE_ROOT)
for g in GAMES:
    halves = sorted(os.path.basename(v)[0] for v in videos if os.path.dirname(v).endswith(g))
    meta = f"{TEST_SRC}/metadata/{os.path.basename(g)}/facts.csv"
    print(f"{'✅' if halves == ['1', '2'] else '⚠️ thiếu hiệp'} hiệp {halves} | metadata {'✅' if os.path.exists(meta) else '❌'} | {g}")
print("model   :", MODEL_NAME, "✅" if os.path.exists(f"{DRIVE_MODEL_DIR}/{MODEL_NAME}/caption/model.pth.tar") else "❌ không thấy checkpoint")
print("phép ép :", MAPPER, "✅" if os.path.exists(MAPPER) else "❌")
print("weights :", TS_WEIGHTS or "❌ KHÔNG THẤY file .pdparams")
assert GAMES and os.path.exists(MAPPER) and TS_WEIGHTS
assert os.path.exists(f"{DRIVE_MODEL_DIR}/{MODEL_NAME}/spotting/model.pth.tar")

## 4. Lấy code từ GitHub và cài thư viện

In [ ]:
if os.path.isdir(f"{WORK_DIR}/.git"):
    !git -C {WORK_DIR} fetch -q origin && git -C {WORK_DIR} reset -q --hard origin/{GIT_BRANCH}
else:
    !git clone -q -b {GIT_BRANCH} {GITHUB_REPO} {WORK_DIR}
%cd {WORK_DIR}
!git log --oneline -1
!grep -m1 "^## v" CHANGELOG.md
!pip install -q SoccerNet av tiktoken transformers wandb pycocoevalcap psutil six websockets
!pip install -q --no-deps git+https://github.com/Maluuba/nlg-eval.git@master

import subprocess
os.environ["WANDB_MODE"] = "offline"

def run(cmd):
    # chạy lệnh, in log trực tiếp; báo lỗi ngay tại cell này nếu lệnh thất bại
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=WORK_DIR)
    for line in p.stdout:
        if "it/s]" in line or "s/it]" in line:
            continue
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}): xem log ngay phía trên")

# models/ trỏ thẳng tới sn-models trên Drive (checkpoint đã train)
if not os.path.islink("models"):
    !rm -rf models
!ln -sfn "{DRIVE_MODEL_DIR}" models

## 5. Video → npy TimeSformer 768 chiều

Mỗi hiệp mất khoảng 5–10 phút trên T4. Kết quả lưu lại trên Drive (`video_test/features/`), chạy lại thì không trích lại.

In [ ]:
import shutil
def sync(src_root, dst_root):
    for f in glob.glob(f"{src_root}/**/*_timesformer768.npy", recursive=True):
        dst = os.path.join(dst_root, os.path.relpath(f, src_root))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copyfile(f, dst)

if os.path.isdir(FEAT_CACHE):
    sync(FEAT_CACHE, TEST_DATA)
try:
    run(["python", "timesformer_extract.py", "--weights", TS_WEIGHTS, "--video_root", TEST_SRC,
         "--video_name", "{half}_224p.mp4", "--out_root", TEST_DATA, "--batch_size", str(TS_BATCH), "--games", *GAMES])
finally:
    sync(TEST_DATA, FEAT_CACHE)
missing = [f"{g}/{h}" for g in GAMES for h in (1, 2) if not os.path.exists(f"{TEST_DATA}/{g}/{h}_timesformer768.npy")]
assert not missing, f"Còn thiếu: {missing}. Chạy lại cell này"

## 6. Phép ép 768 → npy 8576 chiều

Dùng **đúng phép ép đã học** ở notebook train (không học lại). Mỗi hiệp ra `1_/2_baidu_from_timesformer.npy`, kích thước (số giây, 8576).

In [ ]:
run(["python", "feature_mapper.py", "apply", "--soccernet_path", TEST_DATA, "--mapper", MAPPER,
     "--games", *GAMES, "--out_name", FEATURES, "--overwrite"])
import numpy as np
for g in GAMES:
    print([np.load(f"{TEST_DATA}/{g}/{h}_{FEATURES}", mmap_mode="r").shape for h in (1, 2)], g)

## 7. Luồng YES: bắt sự kiện + sinh câu bình luận

Model đã train (`spotting` + `caption`) chạy trên npy 8576 chiều của trận mới: tìm thời điểm sự kiện (NMS), rồi GPT-2 sinh một câu cho mỗi sự kiện. Không có nhãn nên không chấm điểm.
Kết quả: `video_test/output/yes/<trận>/results_dense_captioning.json`.

In [ ]:
run(["python", "infer_match.py", "--model_name", MODEL_NAME, "--features", FEATURES, "--gpt_type", GPT_TYPE,
     "--nms_threshold", str(NMS_THRESHOLD), "--top_k", str(TOP_K), "--out_dir", OUT_YES,
     "--game_dirs", *[f"{TEST_DATA}/{g}" for g in GAMES]])

## 7b. Recall của luồng YES so với sự kiện FIFA (tuỳ chọn)

Trận mới không có nhãn SoccerNet nên so với `events.csv` của FIFA: bàn thắng, dứt điểm, phạt góc, phạm lỗi, việt vị, thẻ, thay người.
FIFA chỉ ghi tới **phút**, nên một sự kiện được tính là "bắt được" nếu có sự kiện YES trong phút đó (±15 giây).

- `recall`: tỉ lệ sự kiện FIFA bắt được. Đây là con số chính.
- `FIFA overlap`: tỉ lệ sự kiện YES nằm gần một sự kiện FIFA. FIFA không ghi mọi pha bóng (ném biên, phát bóng...) nên đây **không phải** precision, chỉ là mức thấp nhất.
- `random`: kết quả nếu đặt **cùng số sự kiện ở thời điểm ngẫu nhiên**. Cửa sổ khớp rộng (~90 giây) nên ngẫu nhiên cũng đạt recall khá cao: model chỉ tốt khi **vượt xa** mức ngẫu nhiên.

Chỉ chạy bắt sự kiện (không sinh câu) cho từng ngưỡng trong `NMS_LIST`, lưu vào `output/yes_nms<ngưỡng>/`, không ghi đè `output/yes`.

In [ ]:
import json, pandas as pd
NMS_LIST = [0.4, 0.5, 0.6, 0.7]
rows = []
for nms in NMS_LIST:
    out = f"{TEST_SRC}/output/yes_nms{nms}"
    run(["python", "infer_match.py", "--model_name", MODEL_NAME, "--features", FEATURES, "--gpt_type", GPT_TYPE,
         "--nms_threshold", str(nms), "--out_dir", out, "--no_caption",
         "--game_dirs", *[f"{TEST_DATA}/{g}" for g in GAMES]])
    for g in GAMES:
        name = os.path.basename(g)
        events = f"{TEST_SRC}/metadata/{name}/events.csv"
        if not os.path.exists(events):
            print("⚠️ không có", events); continue
        res = subprocess.run(["python", "eval_vs_fifa.py", "--yes_dir", f"{out}/{name}", "--events", events],
                             capture_output=True, text=True, cwd=WORK_DIR)
        print(res.stdout.strip() or res.stderr[-2000:])
        line = [l for l in res.stdout.splitlines() if l.startswith("RESULT ")]
        if line:
            rows.append({"NMS": nms, "game": name[:35], **json.loads(line[0][7:])})
table = pd.DataFrame(rows)
display(table.round(3))
table.to_csv(f"{TEST_SRC}/output/recall_vs_fifa.csv", index=False)

## 8. (Bản cũ) Luồng NO: LLM ElevenLabs trong các khoảng trống

**Mặc định bỏ qua** (`RUN_OLD_ELEVENLABS = False`). Luồng NO hiện tại là cell 10, giọng đọc OpenAI ở cell 12.

- Khoảng trống = khoảng giữa hai sự kiện **của luồng YES ở bước 7**.
- Mỗi khoảng trống chọn một fact trong `facts.csv` (metadata FIFA), LLM viết lại thành câu nói. Câu có con số không có trong ngữ cảnh thì dùng câu fact gốc.
- Không train, không cần nhãn.

In [ ]:
if not RUN_OLD_ELEVENLABS:
    print("bỏ qua cell 8 (bản cũ, ElevenLabs). Luồng NO: cell 10, giọng đọc OpenAI: cell 12")
else:
    from google.colab import userdata
    llm = LLM
    try:
        os.environ["ELEVENLABS_API_KEY"] = userdata.get("ELEVENLABS_API_KEY")
    except Exception as e:
        print("⚠️ không đọc được secret ELEVENLABS_API_KEY:", type(e).__name__, "→ chạy chế độ template")
        llm = "template"

    if llm == "elevenlabs":
        # kiểm tra key trước: 401 = key sai / đã xoá / thiếu quyền ElevenAgents
        import requests
        key = os.environ["ELEVENLABS_API_KEY"].strip().strip("\"'").strip()
        os.environ["ELEVENLABS_API_KEY"] = key
        r = requests.get("https://api.elevenlabs.io/v1/convai/agents", params={"page_size": 1},
                         headers={"xi-api-key": key}, timeout=30)
        print(f"kiểm tra key ({len(key)} ký tự, bắt đầu '{key[:3]}'):", r.status_code, "OK" if r.ok else r.text[:300])
        assert r.ok, ("ElevenLabs không nhận key: tạo key mới (bật quyền ElevenAgents), dán lại vào secret "
                      "ELEVENLABS_API_KEY, rồi chạy lại cell này")

    for g in GAMES:
        name = os.path.basename(g)
        if NO_FLOW_GAMES and name not in NO_FLOW_GAMES:
            print("bỏ qua (không có trong NO_FLOW_GAMES):", name)
            continue
        run(["python", "no_flow_llm.py", "--metadata", f"{TEST_SRC}/metadata/{name}", "--video_dir", f"{TEST_SRC}/{g}",
             "--yes_events", f"{OUT_YES}/{name}/results_dense_captioning.json", "--llm", llm, "--llm_model", LLM_MODEL,
             "--agent_cache", f"{DRIVE_MODEL_DIR}/elevenlabs_agent.json", "--out_dir", OUT_TIMELINE])

## 9. (Bản cũ) Timeline YES + NO của cell 8

**Mặc định bỏ qua** (`RUN_OLD_ELEVENLABS = False`).

`[EVENT]`: câu của luồng YES (model). `[INFO ]`: câu của luồng NO (LLM từ metadata).

In [ ]:
if not RUN_OLD_ELEVENLABS:
    print("bỏ qua cell 9 (bản cũ, ElevenLabs)")
else:
    for g in GAMES:
        path = f"{OUT_TIMELINE}/{os.path.basename(g)}/commentary_timeline.txt"
        if not os.path.exists(path):
            print("chưa có timeline:", g)
            continue
        lines = open(path, encoding="utf-8").read().splitlines()
        print(f"\n===== {g}: {sum('[EVENT]' in l for l in lines)} câu YES, {sum('[INFO ]' in l for l in lines)} câu NO")
        print("\n".join(lines[:60]))

## 10. Luồng NO: kịch bản 2.0 + gpt-4o-mini, bình luận liên tục

Kịch bản 2.0 chọn **nói gì, lúc nào** (khuôn câu điền metadata FIFA), LLM **`gpt-4o-mini`** (`NO_LLM = "openai"`) viết thành câu bình luận tự nhiên, vừa thời gian trống. Giọng đọc: `gpt-4o-mini-tts` ở cell 12.

- Theo `NO_FLOW_SCRIPT.md`: **câu bắt buộc** (chào mừng, 2 đội + HLV, ý nghĩa trận, tin nóng bàn thắng / thẻ / thay người, nhắc tỉ số mỗi ~10 phút, hết hiệp 1, đầu hiệp 2, phút 80', bù giờ, hết trận) và **thông tin nền** trong khoảng trống dài.
- Mỗi câu là khuôn có biến điền từ metadata FIFA (dùng cho mọi trận); LLM nói lại tự nhiên. Câu của LLM bị loại (dùng lại khuôn) nếu sai số, có tên lạ, quá dài so với thời gian trống, có lời khen chê ("key player", "dominant"...) hoặc tự gọi một trận cũ là "lần gặp gần nhất".
- Câu YES được bỏ `[PLAYER]` / `[TEAM]` ("the player", "the team") để đọc thành tiếng.
- `CONTINUOUS = True` (v5.5): **bình luận liên tục**. Sau kịch bản, mọi khoảng lặng > 3 giây giữa 2 câu được lấp bằng câu `[FILL]` vừa đúng độ dài khoảng lặng: cầu thủ và số áo, hàng hậu vệ / tiền vệ / tiền đạo, đội trưởng, HLV, sơ đồ, dự bị, phong độ, đối đầu, sân, tỉ số và thời gian còn lại, nhắc lại bàn thắng / thẻ / thay người, đếm phạt góc / phạm lỗi / dứt điểm từ luồng YES ("That's the third corner of the half"). Các chủ đề luân phiên; một câu không lặp lại trong 15 phút cho đến khi đã nói hết.
- Câu `[FILL]` dùng khuôn câu (không gọi LLM: hàng trăm câu mỗi trận) trừ khi `FILL_LLM = True`.
- Cell in ra mỗi hiệp: số phút có lời nói, khoảng lặng dài nhất.
- Kết quả: `video_test/output/timeline_v2/<trận>/commentary_timeline.txt` và `.json`.

In [ ]:
from google.colab import userdata
llm_v2 = NO_LLM
if llm_v2 == "openai":
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY").strip().strip("\"'").strip()
        run(["python", "openai_client.py", "--check", "--model", OPENAI_LLM_MODEL])
    except Exception as e:
        print("⚠️ không dùng được OPENAI_API_KEY:", type(e).__name__, e, "→ câu NO từ khuôn (không LLM)")
        llm_v2 = "template"
try:
    if llm_v2 != "gemini":
        raise LookupError("NO_LLM is not gemini")
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY").strip().strip("\"'").strip()
except Exception as e:
    if llm_v2 == "gemini":
        print("⚠️ không đọc được secret GEMINI_API_KEY:", type(e).__name__, "→ câu NO từ khuôn (không LLM)")
        llm_v2 = "template"
if "GEMINI_API_KEY" in os.environ and llm_v2 == "gemini":
    try:
        run(["python", "gemini_client.py", "--check"])
    except RuntimeError as e:
        print("⚠️ kiểm tra key Gemini lỗi → chạy chế độ template:", e)
        llm_v2 = "template"

OUT_TIMELINE_V2 = f"{TEST_SRC}/output/timeline_v2"
for g in GAMES:
    name = os.path.basename(g)
    if NO_FLOW_V2_GAMES and name not in NO_FLOW_V2_GAMES:
        print("bỏ qua (không có trong NO_FLOW_V2_GAMES):", name)
        continue
    run(["python", "no_flow_v2.py", "--metadata", f"{TEST_SRC}/metadata/{name}", "--video_dir", f"{TEST_SRC}/{g}",
         "--yes_events", f"{OUT_YES}/{name}/results_dense_captioning.json", "--llm", llm_v2,
         "--llm_model", GEMINI_LLM_MODEL if llm_v2 == "gemini" else "none", "--out_dir", OUT_TIMELINE_V2,
         "--openai_model", OPENAI_LLM_MODEL, "--wps", str(WORDS_PER_SECOND)] +
        (["--continuous"] if CONTINUOUS else []) + (["--fill_llm"] if FILL_LLM else []) +
        (["--llm_half", str(DEMO_HALF), "--llm_start", DEMO_START or "00:00"] + (["--llm_end", DEMO_END] if DEMO_END else [])
         if LLM_DEMO_ONLY else []))
    path = f"{OUT_TIMELINE_V2}/{name}/commentary_timeline.txt"
    lines = open(path, encoding="utf-8").read().splitlines()
    no = [l for l in lines if "[EVENT]" not in l and "[MUTED]" not in l and "[FIFA ]" not in l]
    fill = [l for l in no if "[FILL ]" in l]
    print(f"\n===== {name}: {len(lines) - len(no)} câu YES, {len(no)} câu NO ({len(fill)} câu lấp khoảng lặng)")
    if CONTINUOUS:
        t0 = DEMO_START or "00:00"
        part = [l for l in lines if l.startswith(f"{DEMO_HALF} - ") and l[4:9] >= t0
                and (not DEMO_END or l[4:9] < DEMO_END)]
        print(f"Đoạn demo hiệp {DEMO_HALF}, {t0} - {DEMO_END or 'hết'}: {len(part)} câu. 40 câu đầu:")
        print("\n".join(part[:40]))
    else:
        print("\n".join(no))

## 11. (Bản cũ) Giọng đọc: Gemini TTS, ghép vào video

**Mặc định bỏ qua** (`RUN_GEMINI_TTS = False`). Giọng đọc hiện tại: OpenAI `gpt-4o-mini-tts` ở cell 12.

- 2 giọng: `VOICE_YES` đọc câu sự kiện (luồng YES), `VOICE_NO` đọc câu dẫn chương trình (luồng NO). Bàn thắng đọc giọng hào hứng.
- Mỗi câu thành một file wav (lưu cache trên Drive: chạy lại không tốn quota cho câu đã đọc), đặt đúng thời điểm, ghép thành audio từng hiệp, rồi lồng vào video (tiếng gốc của video giảm còn 25%).
- Thời gian (v5.1) theo độ dài thật của từng clip:
  - câu NO nối tiếp (chuỗi A1→A2→A3, C, G, hoặc cách câu trước ≤ 5 s) bắt đầu ở **giây kế tiếp** sau khi câu trước đọc xong;
  - câu YES không bao giờ đọc trước sự kiện; câu trước chưa đọc xong thì lùi đến giây kế tiếp;
  - clip dài hơn khoảng trống đến câu sau thì tăng tốc (không đổi cao độ), tối đa `TTS_MAX_SPEED`.
- `audio_timeline.json`: `gameTime` → `gameTimeEnd` = lúc câu thực sự được nghe; `planned` = giờ trong kịch bản.
- `TTS_TIMELINE = "v1"`: đọc timeline của cell 8 (`output/timeline`), câu `[PLAYER] ([TEAM])` đọc thành "the player (the team)".
- `TTS_MAX_LINES = 30`: thử 30 câu đầu trước. Quota TTS miễn phí có giới hạn mỗi ngày; hết quota thì cell dừng, giữ các câu đã đọc, hôm sau chạy tiếp.
- Kết quả: `video_test/output/timeline_v2/<trận>/audio/1_commentary.mp4`, `2_commentary.mp4` (+ wav).

In [ ]:
if not RUN_GEMINI_TTS:
    print("bỏ qua cell 11 (Gemini TTS). Giọng đọc OpenAI: cell 12")
else:
    for g in GAMES:
        name = os.path.basename(g)
        base = OUT_TIMELINE_V2 if TTS_TIMELINE == "v2" else OUT_TIMELINE
        timeline = f"{base}/{name}/commentary_timeline.json"
        if not os.path.exists(timeline):
            print(f"chưa có timeline {TTS_TIMELINE} (chạy cell {10 if TTS_TIMELINE == 'v2' else 8}):", name)
            continue
        run(["python", "tts_v2.py", "--timeline", timeline, "--video_dir", f"{TEST_SRC}/{g}",
             "--tts_model", GEMINI_TTS_MODEL, "--voice_yes", VOICE_YES, "--voice_no", VOICE_NO,
             "--max_lines", str(TTS_MAX_LINES), "--max_speed", str(TTS_MAX_SPEED)])

    from IPython.display import Audio, display
    for g in GAMES:
        base = OUT_TIMELINE_V2 if TTS_TIMELINE == "v2" else OUT_TIMELINE
        wav = f"{base}/{os.path.basename(g)}/audio/1_commentary.wav"
        if os.path.exists(wav):
            print("Nghe thử 3 phút đầu hiệp 1:", os.path.basename(g))
            import wave, numpy as np
            with wave.open(wav) as w:
                audio = np.frombuffer(w.readframes(w.getframerate() * 180), dtype=np.int16)
                display(Audio(audio, rate=w.getframerate()))

## 12. Giọng đọc OpenAI gpt-4o-mini-tts (luồng YES + NO) → video demo

`TTS_ENGINE` ở cell 2:
- `"openai"` (v5.6): OpenAI TTS (`OPENAI_TTS_MODEL`, mặc định `gpt-4o-mini-tts`), giọng tự nhiên hơn, có chỉ dẫn phong cách (bình luận sự kiện nhanh, bàn thắng phấn khích, dẫn chương trình ấm). Cần key: Colab → biểu tượng 🔑 Secrets → thêm `OPENAI_API_KEY`, bật quyền cho notebook. Key không được ghi ra file. Trả phí theo số ký tự (10 phút bình luận liên tục: khoảng vài nghìn ký tự).
- `OPENAI_STYLE` (v6.2): `"epic"` (mặc định) dặn giọng đọc hào hùng như bình luận viên Anh ở trận lớn, bàn thắng thì gào lên; `"calm"` là cách đọc nhẹ nhàng cũ. Đổi giọng hoặc cách đọc thì các câu được đọc lại từ đầu (tốn lại tiền OpenAI cho đoạn demo).
- Nghe thử trước khi chọn giọng: `!python voice_samples.py --out_dir {TEST_SRC}/output/voice_samples` tạo mỗi giọng một file (cùng 3 câu: sự kiện, bàn thắng, dẫn chương trình).
- `"kokoro"`: Kokoro-82M, model mã nguồn mở **chạy ngay trên Colab** (CPU hoặc GPU): không cần key, không có quota, văn bản không gửi ra ngoài.

Chỉ đọc tiếng Anh.

- Dùng timeline của cell 10 (`TTS_TIMELINE = "v2"`) hoặc cell 8 (`"v1"`). Gemini hết quota thì đặt `GEMINI_LLM_MODEL = "template"` ở cell 2 rồi chạy cell 10: câu NO lấy từ khuôn câu, không gọi LLM.
- 2 giọng: `KOKORO_VOICE_YES` (sự kiện, đọc nhanh hơn 1,1 lần), `KOKORO_VOICE_NO` (dẫn chương trình). Bàn thắng đọc nhanh 1,15 lần.
- Trước khi đọc, ký hiệu được đổi thành lời: `85'` → "85th minute", `2-1` → "2 1", `1-0` → "1 nil", `3-4-2-1` → "3 4 2 1".
- `DEMO_HALF`, `DEMO_START`, `DEMO_END`: chỉ đọc các câu trong đoạn này và cắt riêng video đoạn đó. Cell in ra giờ các bàn thắng để chọn đoạn.
- Kết quả: `video_test/output/timeline_v2/<trận>/audio_<engine>/<hiệp>_demo.mp4` (đoạn demo), `<hiệp>_commentary.mp4` (cả hiệp), `audio_timeline.json`.

In [ ]:
# Kokoro cần Python 3.10–3.12, Colab dùng 3.13 → môi trường Python 3.12 riêng (uv), chỉ tạo một lần mỗi phiên
KOKORO_PY = "/content/kokoro_env/bin/python"
if TTS_ENGINE == "openai" and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY").strip().strip("\"'").strip()
elif not os.path.exists("/content/kokoro_env/.ready"):
    !pip install -q uv
    !uv venv -q --seed --python 3.12 /content/kokoro_env
    !uv pip install -q --python {KOKORO_PY} torch --index-url https://download.pytorch.org/whl/cpu
    !uv pip install -q --python {KOKORO_PY} "kokoro>=0.9.4" soundfile numpy
    !{KOKORO_PY} -m spacy download en_core_web_sm > /dev/null 2>&1
    !apt-get -qq -y install espeak-ng > /dev/null 2>&1
    !{KOKORO_PY} -c "import kokoro, torch; print('Kokoro OK, torch', torch.__version__)" && touch /content/kokoro_env/.ready
import json
from IPython.display import Video, Audio, display

base = OUT_TIMELINE_V2 if TTS_TIMELINE == "v2" and "OUT_TIMELINE_V2" in globals() else (
    f"{TEST_SRC}/output/timeline_v2" if TTS_TIMELINE == "v2" else OUT_TIMELINE)
for g in GAMES:
    name = os.path.basename(g)
    timeline = f"{base}/{name}/commentary_timeline.json"
    if DEMO_GAMES and name not in DEMO_GAMES:
        continue
    if not os.path.exists(timeline):
        print(f"chưa có timeline {TTS_TIMELINE} (chạy cell {10 if TTS_TIMELINE == 'v2' else 8}):", name)
        continue
    goals = [c["gameTime"] for c in json.load(open(timeline, encoding="utf-8"))["comments"] if c.get("topic") == "goal"]
    print(f"===== {name} | bàn thắng (hiệp - phút:giây): {', '.join(goals) or 'không có trong timeline'}")
    out_dir = f"{base}/{name}/audio_{TTS_ENGINE}"
    if TTS_ENGINE == "openai":
        cmd = ["python", "tts_v2.py", "--engine", "openai", "--openai_model", OPENAI_TTS_MODEL,
               "--openai_voice_yes", OPENAI_VOICE_YES, "--openai_voice_no", OPENAI_VOICE_NO, "--rpm", "40",
               "--openai_style", OPENAI_STYLE]
    else:
        cmd = [KOKORO_PY, "-W", "ignore", "tts_v2.py", "--engine", "kokoro",
               "--kokoro_voice_yes", KOKORO_VOICE_YES, "--kokoro_voice_no", KOKORO_VOICE_NO]
    cmd += ["--timeline", timeline, "--video_dir", f"{TEST_SRC}/{g}",
            "--halves", str(DEMO_HALF), "--max_speed", str(TTS_MAX_SPEED), "--out_dir", out_dir]
    if DEMO_START or DEMO_END:
        cmd += ["--start", DEMO_START or "00:00"] + (["--end", DEMO_END] if DEMO_END else [])
    run(cmd)
    demo = f"{out_dir}/{DEMO_HALF}_demo.mp4" if (DEMO_START or DEMO_END) else f"{out_dir}/{DEMO_HALF}_commentary.mp4"
    if os.path.exists(demo):
        size = os.path.getsize(demo) / 1e6
        print(f"video demo: {demo} ({size:.0f} MB)")
        if size < 80:
            display(Video(demo, embed=True, width=640))
        else:
            print("file lớn, mở trên Drive để xem")